In [1]:
import pandas as pd

# Load the datasets
train_df = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/01_bank_customer_churn/train.csv')
test_df = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/01_bank_customer_churn/test.csv')

# Drop the columns that will not be interesting to models
columns_to_drop = ['CustomerId', 'Complain', 'Surname']
train_df_processed = train_df.drop(columns=columns_to_drop).copy()
test_df_processed = test_df.drop(columns=columns_to_drop).copy()

# Display the first few rows of the processed datasets
train_df_processed.head(), test_df_processed.head()


(   RowNumber  CreditScore Geography  ... Satisfaction Score  Card Type  Point Earned
 0       9255          686    France  ...                  2    DIAMOND           510
 1       1562          632   Germany  ...                  4   PLATINUM           959
 2       1671          559     Spain  ...                  4     SILVER           327
 3       6088          561    France  ...                  2     SILVER           567
 4       6670          517    France  ...                  3   PLATINUM           727
 
 [5 rows x 15 columns],
    RowNumber  CreditScore Geography  ... Satisfaction Score  Card Type  Point Earned
 0       6253          596   Germany  ...                  1       GOLD           709
 1       4685          623    France  ...                  2     SILVER           508
 2       1732          601     Spain  ...                  1       GOLD           281
 3       4743          506   Germany  ...                  2     SILVER           979
 4       4522          560  

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info_train = get_column_info(train_df_processed)
print("column_info_train")
print(column_info_train)

column_info_test = get_column_info(test_df_processed)
print("column_info_test")
print(column_info_test)


2025-08-30 18:03:51.082 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info_train
{'Category': ['Geography', 'Gender', 'Card Type'], 'Numeric': ['RowNumber', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Satisfaction Score', 'Point Earned'], 'Datetime': [], 'Others': []}
column_info_test
{'Category': ['Geography', 'Gender', 'Card Type'], 'Numeric': ['RowNumber', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Satisfaction Score', 'Point Earned'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import OneHotEncode

# Initialize the OneHotEncode tool for the columns 'Geography' and 'Card Type'
one_hot_encoder = OneHotEncode(features=['Geography', 'Card Type'])

# Fit the encoder on the training data and transform both the training and test data
train_df_encoded = one_hot_encoder.fit_transform(train_df_processed.copy())
test_df_encoded = one_hot_encoder.transform(test_df_processed.copy())

# Display the first few rows of the encoded datasets
train_df_encoded.head(), test_df_encoded.head()


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


(   RowNumber  CreditScore  ... Card Type_PLATINUM  Card Type_SILVER
 0       9255          686  ...                0.0               0.0
 1       1562          632  ...                1.0               0.0
 2       1671          559  ...                0.0               1.0
 3       6088          561  ...                0.0               1.0
 4       6670          517  ...                1.0               0.0
 
 [5 rows x 20 columns],
    RowNumber  CreditScore  ... Card Type_PLATINUM  Card Type_SILVER
 0       6253          596  ...                0.0               0.0
 1       4685          623  ...                0.0               1.0
 2       1732          601  ...                0.0               0.0
 3       4743          506  ...                0.0               1.0
 4       4522          560  ...                0.0               1.0
 
 [5 rows x 20 columns])

In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Print column information for the latest processed DataFrames
column_info_train = get_column_info(train_df_encoded)
column_info_test = get_column_info(test_df_encoded)

print("Column information for train_df_encoded:")
print(column_info_train)
print("\nColumn information for test_df_encoded:")
print(column_info_test)


Column information for train_df_encoded:
{'Category': ['Gender'], 'Numeric': ['RowNumber', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Satisfaction Score', 'Point Earned', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'Card Type_DIAMOND', 'Card Type_GOLD', 'Card Type_PLATINUM', 'Card Type_SILVER'], 'Datetime': [], 'Others': []}

Column information for test_df_encoded:
{'Category': ['Gender'], 'Numeric': ['RowNumber', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Satisfaction Score', 'Point Earned', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'Card Type_DIAMOND', 'Card Type_GOLD', 'Card Type_PLATINUM', 'Card Type_SILVER'], 'Datetime': [], 'Others': []}


In [5]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# Define the feature columns and target column
feature_columns = [col for col in train_df_encoded.columns if col not in ['RowNumber', 'Exited']]
target_column = 'Exited'

# Prepare the data for training
X_train = train_df_encoded[feature_columns]
y_train = train_df_encoded[target_column]

X_test = test_df_encoded[feature_columns]
y_test = test_df_encoded[target_column]

# Initialize and train the XGBoost model
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Compute the area under the ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
roc_auc


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Gender: object

In [6]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

# Define the feature columns and target column
feature_columns = [col for col in train_df_encoded.columns if col not in ['RowNumber', 'Exited']]
target_column = 'Exited'

# Convert 'Gender' column to numeric using label encoding
label_encoder = LabelEncoder()
train_df_encoded['Gender'] = label_encoder.fit_transform(train_df_encoded['Gender'])
test_df_encoded['Gender'] = label_encoder.transform(test_df_encoded['Gender'])

# Prepare the data for training
X_train = train_df_encoded[feature_columns]
y_train = train_df_encoded[target_column]

X_test = test_df_encoded[feature_columns]
y_test = test_df_encoded[target_column]

# Initialize and train the XGBoost model
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Compute the area under the ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
roc_auc

0.8693898038321529